# HAC Steel OCBF — Phase 0: Seismic Base Shear & Capacity Reconciliation

**ELF base shear + capacity-design demands — ASCE 7-22 (Ch. 12 & 15), AISC 360-22 / 341-22, 2025 CBC**

| field | value |
|---|---|
| Project | OAI Richmond Robotic Lab — Phase 2 |
| Job No. | DG26.0160.00 |
| Location | 1411 Harbour Way S., Richmond, CA |
| Date | 2026-07-29 |
| Subject | HAC Steel OCBF — base shear · **Rev B — per S. Aher call, 2026-07-30** |
| Prepared | Jeffrey (Structural Intern) |
| Checked | S. Aher (pending) |

**Basis.** 2025 CBC / ASCE 7-22 (Ch. 12 & 15); AISC 360-22 & 341-22. The permit
design remains code-based (ELF, Steel OCBF, `R = 3.25`); this sheet supports the
beyond-code PBD options study for the HAC base connection. Units are carried in
kip–in–ksi via `forallpeople`; the seismic coefficients `SDS`, `Cs`, `R`, `Ie`
are expressed in units of `g`.

**Purpose.** Establish the seismic base shear for one typical HAC module and the
amplified (capacity-design) demands on the force-controlled base connection, then
reconcile them against the frame's lateral capacity and the single-rod
fuse mechanism. This is the hand-calc anchor for Phase 0 of the nonlinear
workflow: the 2D Perform-3D pushover must reproduce `V` and the fuse plateau, and
every capacity-protected element must remain elastic against the fuse overstrength.

> **PRELIMINARY — PBD OPTIONS STUDY — NOT FOR PERMIT / CONSTRUCTION**

> **Revision B (per call with S. Aher, 2026-07-30).** Fuse concept locked to a
> **single ductile threaded rod** (one rod + transition couplers, sized by force)
> tying each brace / column to a post-installed anchor base plate; interior and
> brace-base columns are **telescoping legs** dropped into the base plate (≈ ¼″
> plastic-shim gap — tight at the top, looser at the base: transfers shear while
> permitting uplift and small rocking rotation). Overturning is now checked on the
> **whole structure** (RISA dead ≈ 220 kip), and the fuse force is taken from
> **David's RISA reactions (S0.04)**; the hand overturning is a sanity check.
> §§1–7 (per-module ELF) are unchanged and remain the permit basis.


In [1]:
%%capture

# --- Environment setup -------------------------------------------------
!pip install handcalcs forallpeople numpy pandas

import sys, os
sys.path.append(os.path.abspath('..'))     # project root -> makes `helpers` importable

# One import brings: units (kip, inch, ksi, ...), math (sin, cos, pi, ...),
# sig(), export_notebook(), setup_formatting(), and the handcalcs %%render magic.
from helpers.formatting import *
setup_formatting(3)                          # 3 significant figures + kip-inch unit rendering

## 1  Seismic Design Parameters (ASCE 7-22 / 2025 CBC)

Mapped and design spectral values from the Concept BOD (Site Class D, SDC D).
`SDS` governs the short-period plateau; `Ω0` and `Ie` follow Table 15.4-1 for a
Steel OCBF designed as a nonbuilding structure similar to a building.

In [2]:
%%render params
S_S     = 1.960          # g, mapped $MCE_R$ (Ch. 22)
S_1     = 0.680          # g, mapped $MCE_R$
S_DS    = 1.310          # g, design short-period (governs)
S_D1    = 1.130          # g, design 1-second
R       = 3.250          # Steel OCBF, Table 15.4-1
Omega_0 = 2.000          # overstrength factor
I_e     = 1.000          # Risk Category II
T_a     = 0.182          # s, approx. period (Eq. 12.8-7)
W       = 54*kip         # module seismic weight (frame + MEP)


<IPython.core.display.Latex object>

## 2  Period Region Check

The spectral corner period `Ts = SD1 / SDS` locates the module on the response
spectrum.

In [3]:
%%render
T_s = S_D1 / S_DS          # spectral corner period (s)


<IPython.core.display.Latex object>

Since `Ta = 0.182 s  ≪  Ts = 0.863 s`, the module sits on the **short-period
(constant-acceleration) plateau** — it attracts near-peak spectral acceleration.
This is the physical root of the acceleration problem the PBD study addresses.

## 3  Seismic Base Shear (ELF, ASCE 7-22 §12.8)

`Cs` from Eq. 12.8-2, bracketed by the upper limit (Eq. 12.8-3) and the two lower
limits (Eqs. 12.8-5, 12.8-6). Eq. 12.8-6 applies because `S1 = 0.68 g ≥ 0.6 g`.

In [4]:
%%render
C_s     = S_DS / (R / I_e)              # Eq. 12.8-2  (governs)
C_smax  = S_D1 / (T_a * (R / I_e))      # Eq. 12.8-3  upper limit
C_smin1 = 0.044 * S_DS * I_e            # Eq. 12.8-5  lower limit
C_smin2 = 0.5 * S_1 / (R / I_e)         # Eq. 12.8-6  (S1 >= 0.6g)
V       = C_s * W                       # Eq. 12.8-1  base shear


<IPython.core.display.Latex object>

`Cs = 0.403` by Eq. 12.8-2 (`0.105 < 0.403 < 1.910`, so neither limit
governs). **Seismic base shear `V = 21.8 kip` per 26-ft module** — this is the
permit-basis demand (`R = 3.25`) and does not change; everything below is the
beyond-code reconciliation.

## 4  Vertical Seismic Load Effect

In [5]:
%%render
E_v = 0.2 * S_DS          # Eq. 12.4-4a  vertical seismic (x D)


<IPython.core.display.Latex object>

## 5  Overstrength (Capacity-Design) Demand

Force-controlled demand for the brace connections and anchorage.

In [6]:
%%render
V_o  = Omega_0 * V              # Eq. 12.4-7  force-controlled demand
V_mt = 1.5 * Omega_0 * V        # if multi-tiered OCBF (AISC 341 F1)


<IPython.core.display.Latex object>

If the stacked-X frame is classified **multi-tiered** per AISC 341 §F1,
columns, struts, and their connections use `1.5 Ω0 = 3.0` (`Vmt = 65.3 kip`).
Resolve single- vs multi-tiered before finalizing the capacity-protection
checks — it moves the demand by 50%.

## 6  Demand Path — Brace Force & Base Reactions

Per-brace equilibrium at the base: the QE components `VQE` (vertical) and `HQE`
(horizontal) resolve into the brace axial `PQE`; the amplified reactions are the
anchorage / fuse sizing forces.

In [7]:
%%render
V_QE  = 24*kip                       # per-brace vertical (uplift) QE
H_QE  = 9*kip                        # per-brace horizontal QE
theta = 69.4                         # deg, brace angle = atan($V_{QE} / H_{QE}$)
P_QE  = V_QE / sin(theta * pi/180)   # brace axial QE
P_u   = Omega_0 * P_QE               # amplified brace axial
H_u   = Omega_0 * H_QE               # amplified base shear (shear lug)
T_u   = Omega_0 * V_QE               # amplified base uplift (anchorage)


<IPython.core.display.Latex object>

**Cross-check.** The amplified base reactions (`Hu = 18 kip`, `Tu = 48 kip` per brace) match the S0.04 short-direction values (18 / 48). The **long-direction braced frames** are heavier — S0.04 gives `Ω0·Ez` uplift ≈ 101 kip, which governs the fuse (see §9). **David's RISA model governs** these reactions; the hand calc confirms RISA is sane, not replaces it.

## 7  Brace Capacity (frame overstrength vs. demand)

Representative brace `HSS7×4×3/8` (confirm `Ag`, `r` from the member schedule /
AISC Manual). Expected tension yield per AISC 341; compression per AISC 360 §E3.

In [8]:
%%render params
R_y  = 1.400             # Ry, expected/nominal yield ratio (A500 Gr.C HSS)
F_y  = 50*ksi            # nominal yield
A_g  = 6.900*inch**2     # gross area
K    = 1.000             # effective length factor
L_b  = 110.6*inch        # brace unbraced length
r_b  = 1.550*inch        # governing radius of gyration
E    = 29000*ksi         # modulus


<IPython.core.display.Latex object>

In [9]:
%%render
T_yield  = R_y * F_y * A_g                     # expected tension yield (AISC 341)
lambda_c = K * L_b / r_b                        # slenderness KL/r
F_e      = pi**2 * E / lambda_c**2              # Eq. E3-4  elastic buckling stress
F_cr     = 0.658**(F_y / F_e) * F_y             # Eq. E3-2  inelastic buckling
P_n      = F_cr * A_g                           # nominal compression capacity


<IPython.core.display.Latex object>

Both brace capacities (`Tyield ≈ 483 kip`, `Pn ≈ 238 kip`) dwarf the
`≈ 26 kip` demand. The frame's true lateral overstrength is enormous, so a bare
(Solution 1) frame stays elastic to a very high base shear — which is exactly why
a base fuse pays off: it places a controlled yield far below the frame's brute
capacity and caps what reaches the servers.

## 8  Whole-Structure Overturning Check (per S. Aher, 2026-07-30)

§§1–7 give the ELF base shear **per 26-ft module** (permit basis, matches S0.04).
For the **fuse concept**, S. Aher directs an overturning check on the **whole
structure**: take the full HAC seismic weight from David's RISA dead load
(≈ 220 kip — all main + side modules, not one module), apply the ELF shear
`V = Cs·W` as an equivalent lateral force at `0.7 h`, and compare the overturning
to the restoring available from self-weight. (Freebird's per-module takeoff was
54 kip; the RISA whole-structure weight is the right basis here.)

In [10]:
%%render params
W_str = 220*kip          # whole-structure seismic weight, RISA dead (all modules) - confirm w/ David
h_str = 19.42*ft         # top-of-steel height (S3.03 / S3.04)
k_h   = 0.700            # equivalent-force height factor (Sandesh: 0.7h; use 1.0 for full height)
B_ot  = 19.48*ft         # transverse base width, rocking edge-to-edge (S3.04) - confirm


<IPython.core.display.Latex object>

In [11]:
%%render
V_str  = C_s * W_str            # whole-structure ELF base shear (Cs from Sec. 3)
h_eff  = k_h * h_str            # equivalent-lateral-force height
M_ot   = V_str * h_eff          # global overturning moment
M_rest = W_str * B_ot / 2       # restoring moment from self-weight (acts at center)
SR_ot  = M_rest / M_ot          # global stability ratio (> 1 => no net rigid-body uplift)


<IPython.core.display.Latex object>

With `SR ≈ 1.8 > 1`, the HAC is **globally stable in rigid-body overturning** —
self-weight restoring is ~1.8× the overturning demand (still ~1.3× if the force is
applied at full height, `k_h = 1.0`). So the fuse tension is **not** set by global
tipping; it is governed by **local braced-frame uplift** (a chevron / longitudinal
frame lifting its tension leg), taken from David's RISA reactions in §9.
`V_str ≈ 89 kip` is the whole-structure base shear (S. Aher's ~100–120 kip
ballpark; refine with RISA).

## 9  Fuse Rod — Force & Sizing

The fuse is a **single ductile threaded rod** — transition couplers step a larger
rod down to the reduced fuse shank — tying each brace / column to a post-installed
anchor base plate and acting as a pure **axial element** (diameter is the design
variable). It is sized to yield at the seismic uplift *demand*; the base plate,
anchors, gusset, and shear lug are capacity-protected for the fuse *overstrength*
so yielding stays in the rod.

Uplift demands are from David's RISA model (drawing S0.04), reported at
`Ω0 = 2.0`. The fuse yields at the **un-amplified** demand, `T_fuse = TΩ / Ω0`:

In [16]:
%%render params
T_shortO = 48*kip        # short-dir brace uplift, amplified - S0.04 (Omega$_0$ * Ex)
T_longO  = 101*kip       # long-dir braced-frame uplift, amplified - S0.04 (Omega$_0$ * Ez), governs
F_yr     = 105*ksi       # fuse-rod yield (F1554 Gr. 105, per S9.00 anchors)
phi_t    = 0.900         # tension-yield resistance factor


<IPython.core.display.Latex object>

In [13]:
%%render
T_short = T_shortO / Omega_0         # short-dir fuse tension demand
T_long  = T_longO / Omega_0          # long-dir fuse tension demand (governs)
T_fuse  = T_long                      # governing fuse tension demand
A_req   = T_fuse / (phi_t * F_yr)     # required reduced-shank area (tension yield)
d_req   = sqrt(4 * A_req / pi)        # required reduced-shank diameter


<IPython.core.display.Latex object>

Governing fuse demand `T_fuse ≈ 50 kip` → `d_req ≈ 0.83 in`. Select a
**⅞″⌀ reduced shank** (A ≈ 0.60 in², `φ·Fy·A ≈ 57 kip ≥ 50.5`) as the fuse,
stepped to a larger threaded rod through couplers. Short-direction braces need only
`T ≈ 24 kip` — a **¾″⌀ shank** (`φ·Fy·A ≈ 42 kip`) suffices.

**Capacity protection.** The base plate (PL 1″×14½″×12″, S9.00/2), the ¾″⌀ Gr. 105
anchors, gusset, and shear lug are checked against the fuse *overstrength*
(`Ry·Rt` on `A·Fy`), not the demand — so yielding is confined to the rod, and the
rod's yield force is the ceiling on the base shear and hence on floor acceleration.

> **Confirm before sizing.** This fuse force is preliminary — **David's RISA
> per-frame reactions govern**. Grade 105 matches the existing S9.00 anchors but is
> a high-strength / lower-ductility steel; for reliable fuse elongation a more
> ductile grade (A36 / A572-50 turned-down shank) may be preferable. Resolve with
> the connection design.

## 10  Results Summary

In [14]:
import pandas as pd
def kv(x, u="kip"):
    return f"{sig(float(x))} {u}".strip()
mft = lambda x: f"{sig(mag(x)/12.0)} kip\u00b7ft"   # moment in kip-ft (mag() returns kip-inch)

rows = [
    ("Base shear - per module  V",               kv(V),        "R = 3.25 - permit basis - S0.04"),
    ("Overstrength shear  Vo = \u03a90\u00b7V",   kv(V_o),      "single-tier"),
    ("Multi-tier shear  Vmt = 1.5\u00b7\u03a90\u00b7V", kv(V_mt), "if MT per AISC 341 F1"),
    ("Amplified brace axial  Pu",                kv(P_u),      ""),
    ("Amplified base shear  Hu (shear lug)",     kv(H_u),      "S0.04 (18)"),
    ("Amplified base uplift  Tu (anchorage)",    kv(T_u),      "S0.04 (48)"),
    ("Brace tension yield  Tyield",              kv(T_yield),  "AISC 341"),
    ("Brace compression  Pn",                    kv(P_n),      "AISC 360 E3"),
    ("Whole-structure base shear  Vstr",         kv(V_str),    "Wstr ~ 220 kip (RISA)"),
    ("Overturning  Mot = Vstr\u00b70.7h",        mft(M_ot),    "global"),
    ("Restoring  Mrest = Wstr\u00b7B/2",         mft(M_rest),  "self-weight"),
    ("Global stability  SR = Mrest/Mot",         f"{sig(float(SR_ot))}\u00d7", "> 1 -> stable"),
    ("Fuse demand - short dir  Tshort",          kv(T_short),  "S0.04 Ex / \u03a90"),
    ("Fuse demand - long dir  Tlong (governs)",  kv(T_long),   "S0.04 Ez / \u03a90"),
    ("Fuse reduced shank  d_req",                kv(d_req,"in"),"-> select 7/8 in dia"),
]
df = pd.DataFrame(rows, columns=["Quantity", "Value", "Note"])
df


,Quantity,Value,Note
0,Base shear - per module V,21.8 kip,R = 3.25 - permit basis - S0.04
1,Overstrength shear Vo = Ω0·V,43.5 kip,single-tier
2,Multi-tier shear Vmt = 1.5·Ω0·V,65.3 kip,if MT per AISC 341 F1
3,Amplified brace axial Pu,51.3 kip,
4,Amplified base shear Hu (shear lug),18 kip,S0.04 (18)
5,Amplified base uplift Tu (anchorage),48 kip,S0.04 (48)
6,Brace tension yield Tyield,483 kip,AISC 341
7,Brace compression Pn,238 kip,AISC 360 E3
8,Whole-structure base shear Vstr,88.7 kip,Wstr ~ 220 kip (RISA)
9,Overturning Mot = Vstr·0.7h,1210 kip·ft,global


## 11  Reconciliation Targets for the 2D Perform-3D Model

The nonlinear model must reproduce this hand calc before any dynamic result is
trusted:

1. **Elastic period `T1 ≈ 0.18 s`** (modal check — if off, mass or stiffness is wrong).
2. **First yield occurs in the rod fuse**, at the uplift demand (`T_short ≈ 24 kip`,
   `T_long ≈ 50 kip`); the telescoping shim interface transfers shear only.
3. **Pushover plateaus flat** (elastic–plastic axial fuse); a flag shape means
   unintended re-centering.
4. **Force ceiling `≤` fuse overstrength** (`Ry·Rt·A·Fy`) — base plate, anchors,
   gusset, and shear lug stay elastic (capacity-design hierarchy).
5. **Global overturning stable** (`SR ≈ 1.8`); the model must not show rigid-body tipping.
6. **Usage ratios `< 1.0`** on all capacity-protected elements checked against the
   fuse overstrength, per ACI 318-19 Ch. 17 anchorage.

## 12  References & Export

**References:** ASCE 7-22 §12.8, §12.4, Table 15.4-1; AISC 360-22 §E3;
AISC 341-22 §F1; ACI 318-19 Ch. 17. Reaction cross-check per drawing S0.04;
whole-structure overturning basis and single-rod-fuse concept per call with
S. Aher (2026-07-30). Representative brace HSS7×4×3/8 (confirm `Ag`, `r` from the
member schedule). Prepared with `handcalcs` + `forallpeople`.

Export via the helper (writes to the project `outputs/` folder):

In [15]:
# export_notebook("html")   # or "pdf" (WeasyPrint backend on Colab)
export_notebook("html")


Detected active notebook: 'Richmond_HAC_OCBF_BaseShear_CapacityReconciliation_v02.ipynb'
Exporting to HTML in: C:\Users\jeffreyda\hand_calcs\outputs
Success! Your file is ready in the 'outputs' folder.
